In [1]:
import numpy as np
from copy import deepcopy
from ase.io import read
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 


from muonscripts.io_tools.write_pw_input import write_pw_input

from muonscripts.muon_tools.aiida_muon.sites_supercells import gensup
from muonscripts.muon_tools.aiida_muon.sites_supercells import niche_add_impurities

from muonscripts.muesr_tools.sample_muon_sites import anion_sites, target_sites

from muonscripts.symmetry import wyckoff as wyckoff
from muonscripts.symmetry.sites import get_equivalent_sites

In [3]:
import os
dpath='./'

In [4]:
filename = 'UCoGe_pnma.cif'
filename = os.path.join(dpath, filename)
st = Structure.from_file(filename)

print(st)

Full Formula (U4 Co4 Ge4)
Reduced Formula: UCoGe
abc   :   6.845000   4.206000   7.222000
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (12)
  #  SP         a     b       c
---  ----  ------  ----  ------
  0  U     0.0101  0.25  0.7075
  1  U     0.5101  0.25  0.7925
  2  U     0.9899  0.75  0.2925
  3  U     0.4899  0.75  0.2075
  4  Co    0.2887  0.25  0.4172
  5  Co    0.7887  0.25  0.0828
  6  Co    0.7113  0.75  0.5828
  7  Co    0.2113  0.75  0.9172
  8  Ge    0.1967  0.25  0.087
  9  Ge    0.6967  0.25  0.413
 10  Ge    0.8033  0.75  0.913
 11  Ge    0.3033  0.75  0.587


/home/misah/.virtualenvs/pyqm/lib/python3.10/site-packages/pymatgen/io/cif.py:1314: UserWarning: Missing elements Mn, P from PMG structure composition
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):


In [5]:
spacing = 2.0
distance = 1.0

sc_matrix = [
    [2, 0, 0], 
    [0, 3, 0], 
    [0, 0, 2]
    ]

In [6]:
mu_list = niche_add_impurities(
    structure=st,
    niche_atom='H',
    niche_spacing=spacing,
    niche_distance=distance,
    verbose=False,
)
print('Number of muon sites generated:', len(mu_list))
mu_list

Number of muon sites generated: 10


[array([0.001, 0.001, 0.001]),
 array([0.001, 0.001, 0.251]),
 array([0.001, 0.001, 0.501]),
 array([0.251, 0.001, 0.001]),
 array([0.251, 0.001, 0.251]),
 array([0.001     , 0.33433333, 0.001     ]),
 array([0.001     , 0.33433333, 0.251     ]),
 array([0.001     , 0.33433333, 0.501     ]),
 array([0.251     , 0.33433333, 0.251     ]),
 array([0.251     , 0.33433333, 0.751     ])]

In [7]:
np.dot(mu_list, np.linalg.inv(sc_matrix))%1

array([[5.00000000e-04, 3.33333333e-04, 5.00000000e-04],
       [5.00000000e-04, 3.33333333e-04, 1.25500000e-01],
       [5.00000000e-04, 3.33333333e-04, 2.50500000e-01],
       [1.25500000e-01, 3.33333333e-04, 5.00000000e-04],
       [1.25500000e-01, 3.33333333e-04, 1.25500000e-01],
       [5.00000000e-04, 1.11444444e-01, 5.00000000e-04],
       [5.00000000e-04, 1.11444444e-01, 1.25500000e-01],
       [5.00000000e-04, 1.11444444e-01, 2.50500000e-01],
       [1.25500000e-01, 1.11444444e-01, 1.25500000e-01],
       [1.25500000e-01, 1.11444444e-01, 3.75500000e-01]])

In [8]:
# Add positions to original structure
stc = st.copy()
for p in mu_list:
    stc.append('H', p)

filename='UCoGe_initial_muonsites.cif'
print('Writing muon sites to:', filename)
subpath = os.path.join(dpath, 'input_tmpl2')
filename=os.path.join(subpath, filename)
stc.to(fmt='cif', filename=filename)

Writing muon sites to: UCoGe_initial_muonsites.cif


/home/misah/.virtualenvs/pyqm/lib/python3.10/site-packages/pymatgen/core/structure.py:2945: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['U1', 'U1', 'U1', 'U1', 'Co1', 'Co1', 'Co1', 'Co1', 'Ge1', 'Ge1', 'Ge1', 'Ge1', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H20', 'H21']`.
  writer: Any = CifWriter(self, **kwargs)


"# generated using pymatgen\ndata_U2Co2Ge2H5\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   6.84500000\n_cell_length_b   4.20600000\n_cell_length_c   7.22200000\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   U2Co2Ge2H5\n_chemical_formula_sum   'U4 Co4 Ge4 H10'\n_cell_volume   207.92188554\n_cell_formula_units_Z   2\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  U  U1  1  0.01010000  0.25000000  0.70750000  1.0\n  U  U1  1  0.51010000  0.25000000  0.79250000  1.0\n  U  U1  1  0.98990000  0.75000000  0.29250000  1.0\n  U  U1  1  0.48990000  0.75000000  0.20750000  1.0\n  Co  Co1  1  0.28870000  0.25000000  0.41720000  1.0\n  Co  Co1  1  0.78870000  0.250000

In [9]:
equiv_mu_list = get_equivalent_sites(
    fcoords=mu_list,
    host_lattice=st.copy(),
    min_distance=0.5,
    symprec=1e-3,
)
print('Number of equiv muon sites generated:', len(equiv_mu_list))

Number of equiv muon sites generated: 72


In [10]:
# Add positions to original structure
stc = st.copy()
for p in equiv_mu_list:
    stc.append('H', p)

filename='UCoGe_initial_equivmuonsites.cif'
print('Writing equiv muon sites to:', filename)
subpath = os.path.join(dpath, 'input_tmpl2')
filename=os.path.join(subpath, filename)
stc.to(fmt='cif', filename=filename)

Writing equiv muon sites to: UCoGe_initial_equivmuonsites.cif


/home/misah/.virtualenvs/pyqm/lib/python3.10/site-packages/pymatgen/core/structure.py:2945: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['U1', 'U1', 'U1', 'U1', 'Co1', 'Co1', 'Co1', 'Co1', 'Ge1', 'Ge1', 'Ge1', 'Ge1', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24', 'H25', 'H26', 'H27', 'H28', 'H29', 'H30', 'H31', 'H32', 'H33', 'H34', 'H35', 'H36', 'H37', 'H38', 'H39', 'H40', 'H41', 'H42', 'H43', 'H44', 'H45', 'H46', 'H47', 'H48', 'H49', 'H50', 'H51', 'H52', 'H53', 'H54', 'H55', 'H56', 'H57', 'H58', 'H59', 'H60', 'H61', 'H62', 'H63', 'H64', 'H65', 'H66', 'H67', 'H68', 'H69', 'H70', 'H71', 'H72', 'H73', 'H74', 'H75', 'H76', 'H77', 'H78', 'H79', 'H80', 'H81', 'H82', 'H83']`.
  writer: Any = CifWriter(self, **kwargs)


"# generated using pymatgen\ndata_UCoGeH18\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   6.84500000\n_cell_length_b   4.20600000\n_cell_length_c   7.22200000\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   UCoGeH18\n_chemical_formula_sum   'U4 Co4 Ge4 H72'\n_cell_volume   207.92188554\n_cell_formula_units_Z   4\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  U  U1  1  0.01010000  0.25000000  0.70750000  1.0\n  U  U1  1  0.51010000  0.25000000  0.79250000  1.0\n  U  U1  1  0.98990000  0.75000000  0.29250000  1.0\n  U  U1  1  0.48990000  0.75000000  0.20750000  1.0\n  Co  Co1  1  0.28870000  0.25000000  0.41720000  1.0\n  Co  Co1  1  0.78870000  0.25000000  

The grid generated with `spacing=2.0` $\AA$ gave 10 initial muon sites as shown below [with symmetry replicas (b)].

The `spacing` gave grid subdivision $n_x=4, n_y=3, n_z=4$ i.e. divides each lattice length by `spacing` to ensure the sampled points are spaced no further apart than `spacing`

The `distance=1.0` $\AA$ was use to controls pruning cutoff i.e. rejects grid points that lie closer than this threshold to any existing host atom.

<div align="center">
<img src="input_tmpl2/UCoGe_init_muonsites.png" width="800px" />
</div>

In [11]:
supc_list = gensup(
    p_st=st, 
    mu_list=mu_list, 
    sc_mat=sc_matrix,
    only_one_cell = False,
    validate_proximity=True
    )
print('Number of supercells generated:', len(supc_list))
# supc_list

Number of supercells generated: 10


In [12]:
kspacing_density = None # or float, units of A^-1

pseudos = {
    "U": "U.pbe-spfn-rrkjus_psl.1.0.0.UPF",
    "Co": "co_pbe_v1.2.uspp.F.UPF",
    "Ge": "ge_pbe_v1.4.uspp.F.UPF",
    "H": "h_pbe_v1.4.uspp.F.UPF",
}

base_input_data = {
    'CONTROL': {
        'calculation': 'relax',
        'etot_conv_thr': 0.0001,
        'forc_conv_thr': 0.001,
        'max_seconds': 604800,
        'outdir': '/scratch/jaz/ucoge',
        'pseudo_dir': '/home/vol05/scarf795/pseudo/GBRV',
        'restart_mode': 'from_scratch',
        'tprnfor': True,
        'tstress': True,
        'verbosity': 'high',
        'wf_collect': True
    },
    'SYSTEM': {
        'ecutwfc': 60.0,
        'ecutrho': 600.0,
        'degauss': 0.005,
        'occupations': 'smearing',
        'smearing': 'gaussian',
        'tot_charge': 0.0
    },
    'ELECTRONS': 
    {
        'conv_thr': 1e-07,
        'electron_maxstep': 800,
        'mixing_beta': 0.3,
        'mixing_mode': 'local-TF'
    },
    'IONS': {
        'ion_dynamics': 'bfgs',
    },
    'CELL': {
        'cell_dynamics': 'bfgs',
        'cell_factor': 1.0,
        'press': 0.0,
    }
}

for idx, supc in enumerate(supc_list):
    # --- Write Non-Spin-Polarized Input (nspin = 1) ---
    mag_st =  supc.copy()

    input_data_nspin1 = deepcopy(base_input_data)
    prefix = f'UCG_mun0_{idx+1}'
    input_data_nspin1['CONTROL']['prefix'] = prefix

    file_nspin1 = f'{idx+1}.in'
    subpath = os.path.join(dpath, 'input_tmpl2')
    file_nspin1 = os.path.join(subpath, file_nspin1)

    write_pw_input(
        file_nspin1,
        structure=mag_st,
        magmom=None,
        pseudopotentials=pseudos,
        input_data=input_data_nspin1,
        kpts=(2, 2, 2),
        kspacing=kspacing_density,
        koffset=(0, 0, 0),
        crystal_coordinates=True,
    )

    # print(f"  -> {file_nspin1} (prefix='{input_data_nspin1['CONTROL']['prefix']}')")
    prefix_name = input_data_nspin1['CONTROL']['prefix']
    print(f"  -> {file_nspin1:<20} (prefix='{prefix_name}')")

  -> ./input_tmpl2/1.in   (prefix='UCG_mun0_1')
  -> ./input_tmpl2/2.in   (prefix='UCG_mun0_2')
  -> ./input_tmpl2/3.in   (prefix='UCG_mun0_3')
  -> ./input_tmpl2/4.in   (prefix='UCG_mun0_4')
  -> ./input_tmpl2/5.in   (prefix='UCG_mun0_5')
  -> ./input_tmpl2/6.in   (prefix='UCG_mun0_6')
  -> ./input_tmpl2/7.in   (prefix='UCG_mun0_7')
  -> ./input_tmpl2/8.in   (prefix='UCG_mun0_8')
  -> ./input_tmpl2/9.in   (prefix='UCG_mun0_9')
  -> ./input_tmpl2/10.in  (prefix='UCG_mun0_10')


In [13]:
min_species_distances={
    # "U":  2.3,
    # "Co": 2.0,
    # "Ge": 1.6,
} # new

seed_no = 42
n_samples = 1000
t_distance = (0.4, 0.5)


t_coords = [0, 0, 0]

candidate_sites = target_sites(
    st.copy(),
    target_coords=t_coords, 
    n_samples=n_samples,
    target_distance=t_distance,
    min_species_distances=min_species_distances,
    batch_factor=50,
    seed=seed_no,
)

print(
    f"\nGenerated "
    f"{len(candidate_sites)} "
    f"candidate muon sites."
)


Generated 1000 candidate muon sites.


In [14]:
# Add positions to original structure
p_stc = st.copy()
for pos in candidate_sites:
    p_stc.append('H', pos)

file='UCoGe_randomsamples_muonsites.cif'
subpath = os.path.join(dpath, 'input_tmpl2')
file = os.path.join(subpath, file)

p_stc = p_stc.copy()
p_stc = Structure(
    p_stc.lattice, 
    p_stc.species, 
    p_stc.frac_coords,
    site_properties=p_stc.site_properties,   
)

p_stc.to(file)
print(
    f"Saved candidate sites to "
    f"{file}"
)

Saved candidate sites to ./input_tmpl2/UCoGe_randomsamples_muonsites.cif
